## Setup

In [1]:
from dotenv import load_dotenv
from utils import chat_interface

In [2]:
load_dotenv()

True

## Run

In [ ]:
# TODO: Develop your agents under `agentic/agents`
# TODO: Develop your tools under `agentic/tools`
# TODO: Modify `agentic/workflow` in order to orchestrate your agents

In [3]:
# IDEALLY YOUR ONLY IMPORT HERE IS:
# from agentic.workflow import orchestrator

from agentic.workflow import orchestrator

In [4]:
chat_interface(orchestrator, "1")

User: Hi
Assistant: Hello! How can I assist you today?
User: q
Assistant: Goodbye!


In [5]:
list(orchestrator.get_state_history(
    config = {
        "configurable": {
            "thread_id": "1",
        }
    }
))[0].values["messages"]

[HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='43c56100-7a4e-4ff0-adb7-1fbbfdac82e3'),
 AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 60, 'total_tokens': 70, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-C1loRWd5jRqktu5Fut6YZAZsTdc6S', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--0ab48c4a-6130-4dc9-96cd-271e2be7b7c8-0', usage_metadata={'input_tokens': 60, 'output_tokens': 10, 'total_tokens': 70, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]

## Sample Tickets — Successful Resolution (no escalation)

Demonstrates 3 tickets, each routed to a different specialized agent, that get resolved end-to-end without escalating to a human. Uses direct `orchestrator.invoke(...)` calls (not `chat_interface`, which is interactive) so this can run non-interactively.

In [ ]:
import uuid

from langchain_core.messages import HumanMessage, SystemMessage
from data.models import udahub, cultpass
from utils import get_session
from agentic.tools.memory_tools import engine as udahub_engine

# --- Demo 1: knowledge_agent -----
with get_session(udahub_engine) as session:
    user = session.query(udahub.User).filter_by(account_id="cultpass", external_user_id="f556c0").first()
    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id="cultpass",
        user_id=user.user_id,
        channel="chat",
    )
    session.add(ticket)
    session.flush()
    session.add(udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status="open",
        main_issue_type=None,
        tags="subscription, benefits, pricing, access",
    ))
    ticket_id_1 = ticket.ticket_id

config = {"configurable": {"thread_id": ticket_id_1}}
trigger = {
    "messages": [
        SystemMessage(content=f"ThreadId: {ticket_id_1}"),
        HumanMessage(content="What's included in my CultPass subscription?"),
    ],
    "ticket_id": ticket_id_1,
}
result = orchestrator.invoke(trigger, config=config)
print("Escalated:", result["escalated"], "| Finished:", result["finished"])
print("Assistant:", result["messages"][-1].content)


# --- Demo 2: Reservation_agent -----
with get_session(udahub_engine) as session:
    user = session.query(udahub.User).filter_by(account_id="cultpass", external_user_id="f556c0").first()
    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id="cultpass",
        user_id=user.user_id,
        channel="chat",
    )
    session.add(ticket)
    session.flush()
    session.add(udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status="open",
        main_issue_type=None,
        tags="reservation, experience, booking, event, museum, art, exhibition",
    ))
    ticket_id_2 = ticket.ticket_id

config = {"configurable": {"thread_id": ticket_id_2}}
trigger = {
    "messages": [
        SystemMessage(content=f"ThreadId: {ticket_id_2}"),
        HumanMessage(content="I want to book the experience with the title 'Modern Art at MASP'. My user ID is f556c0. Can you help me with that?"),
    ],
    "ticket_id": ticket_id_2,
}
result = orchestrator.invoke(trigger, config=config)
print("Escalated:", result["escalated"], "| Finished:", result["finished"])
print("Assistant:", result["messages"][-1].content)


# --- Demo 3: subscription_agent -----
with get_session(udahub_engine) as session:
    user = session.query(udahub.User).filter_by(account_id="cultpass", external_user_id="f556c0").first()
    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id="cultpass",
        user_id=user.user_id,
        channel="chat",
    )
    session.add(ticket)
    session.flush()
    session.add(udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status="open",
        main_issue_type=None,
        tags="subscription",
    ))
    ticket_id_3 = ticket.ticket_id

config = {"configurable": {"thread_id": ticket_id_3}}
trigger = {
    "messages": [
        SystemMessage(content=f"ThreadId: {ticket_id_3}"),
        HumanMessage(content="Can I cancel my subscription?. My user ID is f556c0. Can you help me with that?"),
    ],
    "ticket_id": ticket_id_3,
}
result = orchestrator.invoke(trigger, config=config)
print("Escalated:", result["escalated"], "| Finished:", result["finished"])
print("Assistant:", result["messages"][-1].content)


    


Supervisor handling escalated ticket c79cb5bf-9178-490c-8713-cd6aa2829a63 for reason: 
Ticket c79cb5bf-9178-490c-8713-cd6aa2829a63 marked as finished and persisted.
Escalated: True | Finished: True
Assistant: I've escalated your request to a supervisor since I couldn't verify your identity or update your subscription status. They will be able to assist you further.
Ticket 4b43d6fb-e58c-4e90-bd1c-1b07eb09f928 marked as finished and persisted.
Escalated: False | Finished: True
Assistant: Your reservation for the experience "Modern Art at MASP" has been successfully created. Your reservation ID is **75e6b6**. Enjoy your experience!
Ticket c53528df-9a33-46df-a654-f777916da3cf marked as finished and persisted.
Escalated: False | Finished: True
Assistant: Your subscription has been successfully canceled. If you need any further assistance, feel free to ask!
